# CNeuroMod QA — scanning timeline

Session acquisition timeline and BOLD scan-duration distributions aggregated from BIDS `*_scans.tsv` (`output_data/scans/`). Figures are written to `output_data/figures/scanning_timeline/`.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
import seaborn as sns

try:
    import ptitprince
    HAS_PTITPRINCE = True
except Exception:
    HAS_PTITPRINCE = False

# Paths are provided by `invoke run-notebooks` as environment variables.
# Figures go in output_data/figures/{FIG_NAME}/ (also the notebook's "already
# ran" sentinel); the metric tables live in output_data/qc_measures|scans/.
FIG_NAME = "scanning_timeline"
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DATA_DIR", "../output_data"))
FIG_DIR = OUTPUT_DIR / "figures" / FIG_NAME
FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_tables(subdir):
    """Concatenate every non-empty TSV in output_data/<subdir>/."""
    frames = []
    for path in sorted((OUTPUT_DIR / subdir).glob("*.tsv")):
        try:
            frame = pd.read_csv(path, sep="\t")
        except (pd.errors.EmptyDataError, OSError):
            continue
        if not frame.empty:
            frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [ ]:
def raincloud(data, x, y, ax, palette="Set2"):
    """RainCloud (ptitprince) with a seaborn violin+strip fallback."""
    data = data[[x, y]].dropna()
    if data.empty:
        ax.set_title(f"no data for {y}")
        return ax
    if HAS_PTITPRINCE:
        try:
            ptitprince.RainCloud(x=x, y=y, hue=x, data=data, palette=palette,
                                 bw=.15, width_viol=.9, ax=ax, orient="v",
                                 alpha=.65, offset=-.05, move=.2, width_box=.1)
            if ax.get_legend():
                ax.get_legend().remove()
            return ax
        except Exception:
            ax.clear()
    sns.violinplot(data=data, x=x, y=y, ax=ax, hue=x, palette=palette,
                   inner=None, cut=0, legend=False)
    sns.stripplot(data=data, x=x, y=y, ax=ax, color="k", size=2, alpha=.4)
    return ax


In [ ]:
scans = load_tables("scans")
if "acq_time" in scans.columns:
    scans["acq_time"] = pd.to_datetime(scans["acq_time"], errors="coerce")
print(f"loaded {len(scans)} scan entries")
if scans.empty:
    print("No scans found — run `invoke run-scans` first (needs data access).")
scans.head()

In [ ]:
if not scans.empty and "acq_time" in scans.columns:
    timed = scans.dropna(subset=["acq_time"]).copy()
    timed["subject"] = timed["subject"].astype(str)
    datasets = sorted(timed["dataset"].dropna().unique())
    cmap = plt.get_cmap("tab20")
    colors = {d: cmap(i / max(len(datasets), 1)) for i, d in enumerate(datasets)}
    fig, ax = plt.subplots(figsize=(20, 8))
    for d in datasets:
        sub = timed[timed.dataset.eq(d)]
        ax.scatter(sub.acq_time, sub.subject, c=[colors[d]], label=d, alpha=.6, s=20)
    ax.set_xlabel("acquisition time")
    ax.set_ylabel("subject")
    ax.legend(title="dataset", bbox_to_anchor=(1.01, 1), loc="upper left")
    fig.savefig(FIG_DIR / "scanning_timeline.png", dpi=120, bbox_inches="tight")

In [ ]:
has_duration = "duration" in scans.columns and scans.get("duration").notna().any()
if not scans.empty and has_duration:
    bold = scans[scans["duration"].notna()].copy()
    bold["subject"] = bold["subject"].astype(str)
    fig, ax = plt.subplots(figsize=(12, 7))
    raincloud(bold, "subject", "duration", ax)
    ax.set_xlabel("subject")
    ax.set_ylabel("scan duration (s)")
    fig.savefig(FIG_DIR / "scan_duration_by_subject.png", dpi=120, bbox_inches="tight")
else:
    print("No usable 'duration' column in scans tables — skipping duration figure.")